🎯 OBJETIVO DA ATIVIDADE

Implementar um modelo de Machine Learning em Google Colab, utilizando o dataset
de voos para prever atrasos de aeronaves.

Objetivos específicos:
- Separar os dados em treino, validação e teste.
- Treinar um modelo preditivo no Colab (por exemplo, XGBoost).
- Avaliar métricas de desempenho (ex.: accuracy, F1, recall, precision).
- Ajustar hiperparâmetros e registrar os impactos nos resultados.
- Realizar inferências com novos dados e analisar os resultados.

🧩 DESAFIO

Utilizando o Google Colab, implemente um pipeline completo de treinamento de
modelo para prever atrasos de voos.
O seu notebook deve, no mínimo:
1. Preparar os dados para o treinamento, dividindo-os entre treino, validação e
teste.
2. Configurar e treinar um modelo de Machine Learning (ex.: XGBoost).
3. Ajustar hiperparâmetros para otimizar o desempenho do modelo.
4. Avaliar os resultados utilizando métricas adequadas.

In [31]:
# Importação de bibliotecas
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import pandas as pd

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [32]:
# Carregando arquivo dataset
df = pd.read_csv("flights_delays_120.csv")

# Informações básicas do dataset
print(f"--- Cabeçalho do Dataset ---\n{df.head()}")
print(f"--- Informações Básicas ----\n{df.info}")
print(f"--- Tipo do Dataset ---\n{df.dtypes}")

# Separando apenas a coluna de interesse
def delayed_column(dataframe):
  candidatos = ["delayed", "atrasos", "alvo", "target", "y"]
  for c in candidatos:
    if c in dataframe.columns:
      return c

  # Caso nenhuma coluna corresponda à busca, levanta um erro
  print(f"ERRO: Nenhuma coluna correspondeu a busca.")
  raise KeyError("Coluna alvo não encontrada.")

--- Cabeçalho do Dataset ---
     airline origin destination  departure_hour  day_of_week weather  delayed
0  TravelAir    GIG         FOR              11            5   Storm        0
1   JetCloud    CNF         SSA              11            3    Wind        0
2   SkyWings    POA         SSA               4            5     Fog        1
3   JetCloud    BSB         FOR               6            4   Storm        1
4   JetCloud    GRU         FOR               3            1    Rain        1
--- Informações Básicas ----
<bound method DataFrame.info of        airline origin destination  departure_hour  day_of_week weather  delayed
0    TravelAir    GIG         FOR              11            5   Storm        0
1     JetCloud    CNF         SSA              11            3    Wind        0
2     SkyWings    POA         SSA               4            5     Fog        1
3     JetCloud    BSB         FOR               6            4   Storm        1
4     JetCloud    GRU         FOR         

In [33]:
# Preparando os dados
alvo = delayed_column(df)

# Separando as colunas em feature (X) e alvo (Y)
x = df.drop(columns=[alvo])
y = df[alvo]

# Separando as colunas entre categóricas e numéricas
cat = [c for c in x.columns if x[c].dtype == "object"]
num = [c for c in x.columns if c not in cat]

print(f"Colunas categóricas: {cat}")
print(f"Colunas numéricas: {num}")

# Dividindo os dados entre treino, validação e teste
x_temp, x_test, y_temp, y_test = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=7
)

# 60% treino, 20% validação e 20% teste
x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=7
)

print(f"Dados divididos: {len(y_train)}, para treino; {len(y_test)} para teste")

Colunas categóricas: ['airline', 'origin', 'destination', 'weather']
Colunas numéricas: ['departure_hour', 'day_of_week']
Dados divididos: 72, para treino; 24 para teste


In [34]:
# Pré-processador e Pipeline do modelo com OneHotEncoder
preprocessador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), cat),
    ("numericas", "passthrough", num)
])

# Criando um modelo XGBoost
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=7
)

pipeline = Pipeline([
    ("prep", preprocessador),
    ("xgb", model)
])

# Treinando o modelo
pipeline.fit(x_train, y_train)
print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [35]:
# Avaliação do modelo com métricas
probas = pipeline.predict_proba(x_test)[:,1]
threshold = 0.5

y_pred = (probas >= threshold).astype(int)

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
print(f"Matriz de confusão:\n{cm}")

# Principais métricas
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall (sensibilidade): {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred, zero_division=0):.4f}")

# Relatório de classificação
print(f"\n--- Relatório de Classificação ---\n{classification_report(y_test, y_pred)}")


Matriz de confusão:
[[14  0]
 [ 0 10]]
Acurácia: 1.0000
Precisão: 1.0000
Recall (sensibilidade): 1.0000
F1-Score: 1.0000

--- Relatório de Classificação ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        10

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [36]:
# Ajuste de hiperparâmetros
param_grid = {
    "xgb__max_depth": [4,6,8],
    "xgb__learning_rate": [0.01,0.05,0.1],
    "xgb__n_estimators": [100,200,300]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1"
)

grid.fit(x_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('categoricas',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['airline',
                                                                          'origin',
                                                                          'destination',
                                                                          'weather']),
                                                                        ('numericas',
                                                                         'passthrough',
                                                                         ['departure_hour',
                                                                          'day_of_week'])])),
                                       ('xgb',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_byno...
                                                      max_cat_threshold=None,
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=6,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=300,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             param_grid={'xgb__learning_rate': [0.01, 0.05, 0.1],
                         'xgb__max_depth': [4, 6, 8],
                         'xgb__n_estimators': [100, 200, 300]},
             scoring='f1')

In [37]:
# Inferência com novos dados para validar previsões
novo_voo = x_test.iloc[[0]]

pipeline.predict(novo_voo)
pipeline.predict_proba(novo_voo)

array([[0.15324295, 0.84675705]], dtype=float32)